# 03 — Error analysis (validation 2022–2023)

Where does the model fail, and when should it keep its mouth shut?

Inputs: `reports/predictions_valid.parquet` (run `uv run python -m octagon evaluate` first).
All analysis uses the LR+GBM blend — the best skill model from M4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from octagon.config import resolve_path

df = pd.read_parquet(resolve_path("reports/predictions_valid.parquet"))
p = df["p_blend"].to_numpy()
y = df["label"].to_numpy(dtype=float)
correct = (p > 0.5) == (y == 1)
conf = np.maximum(p, 1 - p)
print(f"{len(df)} validation fights, overall accuracy {correct.mean():.3f}")

In [ ]:
# Does confidence mean anything? (It should, if calibration is honest.)
buckets = pd.cut(conf, [0.5, 0.55, 0.6, 0.65, 0.7, 1.0])
table = pd.DataFrame({"bucket": buckets, "correct": correct}).groupby(
    "bucket", observed=True)["correct"].agg(["mean", "size"])
table.columns = ["accuracy", "n fights"]
table

In [ ]:
# The abstain band: refuse to call fights the model sees as coin flips.
# Chosen band (config.yaml): [0.45, 0.55].
rows = []
for w in np.arange(0, 0.16, 0.01):
    keep = conf >= 0.5 + w
    rows.append({"band": f"±{w:.2f}", "coverage": keep.mean(),
                 "accuracy on called fights": correct[keep].mean()})
sweep = pd.DataFrame(rows).set_index("band")
ax = sweep.plot(secondary_y="coverage", figsize=(9, 4),
                title="abstain-band tradeoff: call fewer fights, call them better")
plt.show()
sweep.iloc[[0, 3, 5, 8, 10]]

In [ ]:
# The market-disagreement test — the single most important honesty check.
# If the model had exploitable knowledge, it would win its disagreements.
priced = df[df["p_market"].notna()].copy()
mkt, pb = priced["p_market"].to_numpy(), priced["p_blend"].to_numpy()
yy = priced["label"].to_numpy(dtype=float)
agree = (pb > 0.5) == (mkt > 0.5)
model_right = (pb > 0.5) == (yy == 1)
market_right = (mkt > 0.5) == (yy == 1)
print(f"priced fights: {len(priced)}")
print(f"market upset rate: {1 - market_right.mean():.3f}")
print(f"blend agrees with market on {agree.mean():.1%} of fights")
print(f"when they DISAGREE (n={(~agree).sum()}): model right {model_right[~agree].mean():.1%},"
      f" market right {market_right[~agree].mean():.1%}")

In [ ]:
# Segment error rates: debuts, title fights, long layoffs.
exp = np.minimum(df["f1_n_fights"], df["f2_n_fights"])
segments = {
    "debut on card (min fights = 0)": exp == 0,
    "both experienced (min fights 9+)": exp >= 9,
    "title fights": df["title_fight"] == 1,
    "long layoff >1y (either)": (df["f1_layoff_days"] > 365) | (df["f2_layoff_days"] > 365),
}
pd.DataFrame([{"segment": k, "accuracy": correct[np.asarray(v)].mean(),
               "n": int(np.asarray(v).sum())} for k, v in segments.items()]).set_index("segment")

## Takeaways

1. **Confidence is real**: accuracy climbs monotonically from ~53% (near coin-flip calls) to ~77% (confident calls). The calibration work paid off.
2. **Abstain band [0.45, 0.55]** (now in config.yaml): inside it the model is right 53% of the time — statistically a shrug. Refusing those ~24% of fights lifts accuracy on called fights to ~64%.
3. **The market wins the disagreements** (~61% to ~39%). This is the punchline of the whole project so far: the model contains no knowledge the market lacks, and the M6 backtest should be expected to lose money. That expectation, stated *before* running the backtest, is what intellectual honesty looks like.
4. **No debut curse**: fights with debutants aren't harder for the model — likely because matchmaking and the veteran's known record carry signal. Title fights are slightly worse (n=41, noisy).
